In [1]:
import torch
import torchvision
import torch.nn as nn
import os, numpy as np, matplotlib.pyplot as plt, pandas as pd, cv2, random
from PIL import Image
from model import BrainTumorModel
from utils import CLAHETransform, PretrainingDataset

# Pretraining the model on classification and reconstructing

In [2]:
# Keep geometric and photometric parts separate to preserve spatial alignment
geom_transform = torchvision.transforms.Compose([
    torchvision.transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    torchvision.transforms.RandomRotation(degrees=15),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
])

photo_transform = torchvision.transforms.Compose([
    torchvision.transforms.ColorJitter(brightness=0.08, contrast=0.08),
    torchvision.transforms.RandomApply([CLAHETransform()], p=0.5),
])

base_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Instantiate the Pretraining Dataset & Loader
pretrain_dataset = PretrainingDataset(
    root=os.path.join('Data', 'classification', 'Training'),
    geom_transform=geom_transform,
    photo_transform=photo_transform,
    base_transform=base_transform
)

pretrain_loader = torch.utils.data.DataLoader(
    pretrain_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

In [3]:
test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size = (224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
testing_dataset = torchvision.datasets.ImageFolder(root=os.path.join('Data', 'classification', 'Testing'), transform=test_transform)
testing_loader = torch.utils.data.DataLoader(testing_dataset, batch_size=32, shuffle=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Initialize the model with out_channels=3 for RGB reconstruction
model = BrainTumorModel(num_classes=4, out_channels=1)
model.to(device)

# 2. Define the loss criteria
criterion_cls = nn.CrossEntropyLoss()
criterion_recon = nn.L1Loss()

base_lr = 5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.05)
scaler = torch.amp.GradScaler()

num_epochs = 30
log_interval = 10
warmup_ratio = 0.05  # proportion of total steps used for linear warmup

# compute total steps (batches) to build the schedule
total_steps = num_epochs * len(pretrain_loader)
warmup_steps = max(1, int(total_steps * warmup_ratio))

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, total_steps - warmup_steps),
    eta_min=5e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)

history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

global_step = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    # 3. Modify loader unpacking to receive augmented images, clean target images, and labels
    for batch_idx, (augmented_images, clean_images, labels) in enumerate(pretrain_loader, 1):
        augmented_images = augmented_images.to(device)
        clean_images = clean_images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu'):
            # Forward pass: extract classification and reconstruction outputs
            cls_output, reconstruction_output = model(augmented_images)
            
            # Calculate classification and reconstruction losses
            cls_loss = criterion_cls(cls_output, labels)
            reconstruction_loss = criterion_recon(torch.mean(reconstruction_output, dim = 1), torch.mean(clean_images, dim = 1))
            
            # Combine losses (10.0 weight scales the reconstruction loss to a comparable scale)
            loss = cls_loss + 10.0 * reconstruction_loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # step scheduler per batch (after optimizer.step)
        scheduler.step()
        global_step += 1

        train_loss += loss.item() * augmented_images.size(0)
        
        # Calculate training accuracy metrics on classification predictions
        _, preds = torch.max(cls_output, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        if batch_idx % log_interval == 0 or batch_idx == len(pretrain_loader):
            batch_loss = train_loss / train_total
            batch_acc = train_correct / train_total
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch + 1}/{num_epochs} | Batch {batch_idx}/{len(pretrain_loader)} | "
                    f"Train Loss: {batch_loss:.4f} | Train Acc: {batch_acc:.4f} | LR: {current_lr:.6e}")

    epoch_loss = train_loss / train_total
    epoch_acc = train_correct / train_total

    # 4. Validation Loop
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in testing_loader:
            images = images.to(device)
            labels = labels.to(device)
            with torch.amp.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu'):
                # Unpack the classification outputs and discard the reconstruction output during validation
                cls_outputs, _ = model(images)
                loss = criterion_cls(cls_outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(cls_outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch + 1}/{num_epochs} completed | "
            f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

history_df = pd.DataFrame(history)
print('\nTraining history:')
print(history_df)

/home/thoal/anaconda3/envs/pytorch/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Epoch 1/30 | Batch 10/179 | Train Loss: 12.2785 | Train Acc: 0.4406 | LR: 6.679104e-05
Epoch 1/30 | Batch 20/179 | Train Loss: 10.5357 | Train Acc: 0.5953 | LR: 8.358209e-05
Epoch 1/30 | Batch 30/179 | Train Loss: 9.3105 | Train Acc: 0.6823 | LR: 1.003731e-04
Epoch 1/30 | Batch 40/179 | Train Loss: 8.4142 | Train Acc: 0.7273 | LR: 1.171642e-04
Epoch 1/30 | Batch 50/179 | Train Loss: 7.6962 | Train Acc: 0.7538 | LR: 1.339552e-04
Epoch 1/30 | Batch 60/179 | Train Loss: 7.1157 | Train Acc: 0.7781 | LR: 1.507463e-04
Epoch 1/30 | Batch 70/179 | Train Loss: 6.6279 | Train Acc: 0.7982 | LR: 1.675373e-04
Epoch 1/30 | Batch 80/179 | Train Loss: 6.2226 | Train Acc: 0.8133 | LR: 1.843284e-04
Epoch 1/30 | Batch 90/179 | Train Loss: 5.9060 | Train Acc: 0.8222 | LR: 2.011194e-04
Epoch 1/30 | Batch 100/179 | Train Loss: 5.6244 | Train Acc: 0.8331 | LR: 2.179104e-04
Epoch 1/30 | Batch 110/179 | Train Loss: 5.4016 | Train Acc: 0.8395 | LR: 2.347015e-04
Epoch 1/30 | Batch 120/179 | Train Loss: 5.2132 | 

/home/thoal/anaconda3/envs/pytorch/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 2/30 | Batch 90/179 | Train Loss: 2.7836 | Train Acc: 0.9524 | LR: 5.000000e-04
Epoch 2/30 | Batch 100/179 | Train Loss: 2.7907 | Train Acc: 0.9528 | LR: 4.999943e-04
Epoch 2/30 | Batch 110/179 | Train Loss: 2.8100 | Train Acc: 0.9514 | LR: 4.999791e-04
Epoch 2/30 | Batch 120/179 | Train Loss: 2.8051 | Train Acc: 0.9510 | LR: 4.999545e-04
Epoch 2/30 | Batch 130/179 | Train Loss: 2.8036 | Train Acc: 0.9507 | LR: 4.999204e-04
Epoch 2/30 | Batch 140/179 | Train Loss: 2.8091 | Train Acc: 0.9498 | LR: 4.998769e-04
Epoch 2/30 | Batch 150/179 | Train Loss: 2.8061 | Train Acc: 0.9490 | LR: 4.998238e-04
Epoch 2/30 | Batch 160/179 | Train Loss: 2.7968 | Train Acc: 0.9496 | LR: 4.997614e-04
Epoch 2/30 | Batch 170/179 | Train Loss: 2.8045 | Train Acc: 0.9485 | LR: 4.996894e-04
Epoch 2/30 | Batch 179/179 | Train Loss: 2.8002 | Train Acc: 0.9492 | LR: 4.996166e-04
Epoch 2/30 completed | Train Loss: 2.8002 | Train Acc: 0.9492 | Val Loss: 0.0625 | Val Acc: 0.9825
Epoch 3/30 | Batch 10/179 | Trai

In [17]:
test_image, label = next(iter(testing_loader))
test_image = test_image.to('cuda')


In [18]:
cls_output, reg_output = model(test_image)

In [19]:
reconstructed_image = reg_output[0]


In [21]:
test_image

tensor([[[[-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665],
          [-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665],
          [-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665],
          ...,
          [-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665],
          [-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665],
          [-2.0665, -2.0665, -2.0665,  ..., -2.0665, -2.0665, -2.0665]],

         [[-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832],
          [-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832],
          [-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832],
          ...,
          [-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832],
          [-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832],
          [-1.9832, -1.9832, -1.9832,  ..., -1.9832, -1.9832, -1.9832]],

         [[-1.7522, -1.7522, -1.7522,  ..., -1.7522, -1.7522, -1.7522],
          [-1.7522, -1.7522, -